# Day 6: Memory

LangChain in 10 Days · KGP Talkie · Laxmi Kant Tiwari · kgptalkie.com

**Today:** prove the model has no memory, then build some by hand, then let a checkpointer do it. Measure what remembering costs, trim or summarise to afford it, and split short-term memory from long-term.

> Today makes more API calls than any other day. A **429 `free-models-per-min`** is the free tier's limit, not your code. Wait a minute and rerun.

In [1]:
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv()

model = ChatOpenRouter(model="nvidia/nemotron-3-super-120b-a12b:free", temperature=0)

## 1. Prove the model remembers nothing

Two calls, one after the other. The second one asks about the first.

In [2]:
model.invoke("My name is Priya.").text

"Hello Priya! It's lovely to meet you. 😊 How can I assist you today? Whether you have a question, need help with something, or just want to chat—I'm here for you. What's on your mind?"

In [3]:
model.invoke("What is my name?").text

'I don\'t have access to personal information like your name unless you\'ve shared it with me in this conversation. Since this is our first interaction (or I don\'t recall a prior exchange where you told me your name), I genuinely don\'t know what it is.\n\nIf you\'d like to tell me your name, I\'d be happy to use it! Just say something like:  \n*"My name is [Your Name]."*  \n\nOtherwise, I\'ll keep referring to you as "you" or friend unless you specify otherwise. 😊  \nHow can I assist you today?'

It doesn't know. Nothing was stored, each `invoke` is a separate request to a server that keeps nothing. The model is a function: text in, text out.

Everything today is about **you** doing the remembering, then handing that job off.

## 2. Keep the conversation yourself, in a list

Put the conversation in the request: a list, appended to.

In [4]:
from langchain.messages import HumanMessage

history = []

def say(text):
    history.append(HumanMessage(text))
    reply = model.invoke(history)
    history.append(reply)
    return reply.text

In [5]:
say("My name is Priya and I use a wheelchair.")

"Hello Priya! Thank you for sharing that about yourself. It's lovely to meet you. I appreciate you letting me know — it helps me understand your context better, though please know I'm here to chat about whatever you'd like, whether related to your wheelchair use or not. \n\nIf you ever want to discuss accessibility, adaptive tools, experiences navigating the world, or anything else entirely (hobbies, interests, questions, just chatting), I'm here to listen and help however I can. How's your day going so far? 😊"

In [6]:
history

[HumanMessage(content='My name is Priya and I use a wheelchair.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello Priya! Thank you for sharing that about yourself. It's lovely to meet you. I appreciate you letting me know — it helps me understand your context better, though please know I'm here to chat about whatever you'd like, whether related to your wheelchair use or not. \n\nIf you ever want to discuss accessibility, adaptive tools, experiences navigating the world, or anything else entirely (hobbies, interests, questions, just chatting), I'm here to listen and help however I can. How's your day going so far? 😊", additional_kwargs={'reasoning_content': 'Okay, the user just shared that their name is Priya and they use a wheelchair. This seems like a simple self-introduction, but I should be careful not to make assumptions. \n\nHmm, they didn\'t ask a specific question - just stated a fact about themselves. Maybe they\'re testing if I\'ll respond appropriately 

In [7]:
say("Book me a table for two on Friday.")

'I appreciate you asking, Priya! However, I need to be transparent: **I don’t have the ability to make real-world reservations, book tables, or access external systems like restaurant booking platforms** (such as OpenTable, Resy, or direct restaurant lines). My capabilities are limited to providing information, suggestions, and guidance within this chat.\n\nThat said, I’d be happy to help you prepare for booking it yourself! To make the process smooth, here’s what you might need:\n\n1. **Key details to decide first**:  \n   - Exact date (this coming Friday? or a specific date?)  \n   - Preferred time (e.g., 7:00 PM)  \n   - Cuisine type or restaurant name (if you have one in mind)  \n   - Any accessibility needs beyond wheelchair access (e.g., accessible restrooms, space for maneuvering, quiet area)  \n\n2. **When booking, remember to**:  \n   - **Call the restaurant directly** (often best for confirming accessibility specifics) or use their online booking system *if* they have one.  \

It knows the name now. How much it picks up varies, that the first message was **there at all** is what you controlled.

In [8]:
history

[HumanMessage(content='My name is Priya and I use a wheelchair.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello Priya! Thank you for sharing that about yourself. It's lovely to meet you. I appreciate you letting me know — it helps me understand your context better, though please know I'm here to chat about whatever you'd like, whether related to your wheelchair use or not. \n\nIf you ever want to discuss accessibility, adaptive tools, experiences navigating the world, or anything else entirely (hobbies, interests, questions, just chatting), I'm here to listen and help however I can. How's your day going so far? 😊", additional_kwargs={'reasoning_content': 'Okay, the user just shared that their name is Priya and they use a wheelchair. This seems like a simple self-introduction, but I should be careful not to make assumptions. \n\nHmm, they didn\'t ask a specific question - just stated a fact about themselves. Maybe they\'re testing if I\'ll respond appropriately 

Four messages: two from you, two from the model. Everything below is this same list, with the bookkeeping done for you, or made cheaper.

## 3. Hand the list to a checkpointer

`create_agent` with a **checkpointer** does the appending for you.

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

assistant = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a helpful restaurant booking assistant.",
    checkpointer=InMemorySaver(),
)

`tools=[]` for now, Day 7 fills that in. A checkpointer needs to know **which** conversation it's saving, that's `thread_id`.

In [11]:
model.invoke('you pass your message')

assistant.invoke({'messages': ['message']})

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [ ]:
priya = {"configurable": {"thread_id": "priya"}}
assistant.invoke({"messages": [HumanMessage('my name is priya')]}, config=priya)

{'messages': [HumanMessage(content='my name is priya', additional_kwargs={}, response_metadata={}, id='c997b1f5-ba7f-490f-9fc3-6662efd91387'),
  AIMessage(content="Hello Priya! Nice to meet you. I'm here to help you with restaurant bookings, menu recommendations, or anything else you need. How can I assist you today?", additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "my name is priya"\n   - This is a simple personal introduction.\n\n2.  **Identify Role/Context:**\n   - I am a "helpful restaurant booking assistant" (as per system prompt).\n   - The user is providing their name, likely as part of a booking process or general interaction.\n\n3.  **Determine Appropriate Response:**\n   - Acknowledge the name.\n   - Be friendly and professional.\n   - Since I\'m a restaurant booking assistant, I can offer help with bookings, menus, etc.\n   - Keep it brief and welcoming.\n\n4.  **Draft Response:**\n   - "Hello Priya! Nice 

In [ ]:
priya = {"configurable": {"thread_id": "priya"}}

def chat(text, config):
    result = assistant.invoke({"messages": [HumanMessage(text)]}, config=config)
    return result["messages"][-1].text

In [ ]:
chat("My name is Priya and I use a wheelchair.", priya)

"Thank you for sharing, Priya! I'll keep your accessibility needs in mind when finding restaurants for you.\n\nTo help you book a great spot, could you let me know:\n- **City or neighborhood** you'd like to dine in\n- **Date and time** you're planning to go\n- Any **cuisine preferences** or budget range\n\nI'll search for restaurants with full wheelchair accessibility, including ramp-free entrances, accessible restrooms, adequate table spacing, and comfortable seating. Looking forward to helping you out! 😊"

In [ ]:
chat("Book me a table for two on Friday.", priya)

"I'd love to help you book a table, Priya! To find the best wheelchair-accessible restaurant for two on Friday, I just need a few more details:\n\n- **Which city or neighborhood** are you looking in?\n- **What time** on Friday would work best?\n- Any **cuisine preferences** or budget range?\n\nOnce I have those, I’ll search for fully accessible spots with ramp-free entrances, accessible restrooms, and comfortable seating. Just let me know! 😊"

No `history` variable anywhere in your code. The checkpointer kept it.

## 4. thread_id separates one conversation from another

Same assistant, same checkpointer, different `thread_id`.

In [ ]:
rohit = {"configurable": {"thread_id": "rohit"}}

chat("What is my name?", rohit)

"I don't know your name! As an AI, I don't have access to your personal information unless you share it with me. \n\nIf you'd like, you can tell me your name, and I'll keep it in mind for this conversation. Or if you're here to make a restaurant reservation, I'm happy to help with that too! 😊"

In [ ]:
chat("What is my name?", priya)

'Your name is Priya, as you mentioned earlier! 😊 Is there anything else I can help you with for your Friday booking?'

Rohit gets nothing, Priya's thread still knows.

`thread_id` is the entire privacy boundary between two conversations, and it's **a string you chose**. Get it wrong and one user reads another's chat.

## 5. Inspect the saved conversation with get_state

In [ ]:
state = assistant.get_state(priya)

state

In [ ]:
for m in state.values["messages"]:
    m.pretty_print()

================================ Human Message =================================

my name is priya
================================== Ai Message ==================================

Hello Priya! Nice to meet you. I'm here to help you with restaurant bookings, menu recommendations, or anything else you need. How can I assist you today?
================================ Human Message =================================

My name is Priya and I use a wheelchair.
================================== Ai Message ==================================

Thank you for sharing, Priya! I'll keep your accessibility needs in mind when finding restaurants for you.

To help you book a great spot, could you let me know:
- **City or neighborhood** you'd like to dine in
- **Date and time** you're planning to go
- Any **cuisine preferences** or budget range

I'll search for restaurants with full wheelchair accessibility, including ramp-free entrances, accessible restrooms, adequate table spacing, and comfortable se

## 6. Measure how input tokens grow every turn

Every turn sends the **whole** list again, not a figure of speech. The reply carries the token counts.

In [ ]:
import time

usage = []

def chat_tracked(text, config):
    # time.sleep(6)                 # the free tier limits calls per minute
    result = assistant.invoke({"messages": [HumanMessage(text)]}, config=config)
    last = result["messages"][-1]
    usage.append(last.usage_metadata["input_tokens"])
    return last.text

The `sleep` isn't part of the lesson, it just keeps back-to-back turns from tripping the free tier's per-minute limit.

In [ ]:
long_chat = {"configurable": {"thread_id": "cost-demo"}}

chat_tracked("I want to book a table.", long_chat)
chat_tracked("Make it Friday.", long_chat)
chat_tracked("For four people.", long_chat)
chat_tracked("Around eight in the evening.", long_chat)

usage

[31, 133, 265, 415]

Four short questions, and the input grew every turn. Imagine forty turns on a paid model: the last message costs the most and says the least.

Two ways out, and they lose different things.

## 7. Fix one: drop old messages with trim_messages

`trim_messages` walks the list from the end backwards and cuts when the budget runs out. The simplest budget is a **message count**: pass `len` as the counter.

In [ ]:
from langchain.messages import trim_messages

full = assistant.get_state(long_chat).values["messages"]

full

In [ ]:
kept = trim_messages(
    full,
    max_tokens=4,
    token_counter=len,
    strategy="last",
    start_on="human",
)

kept

In [ ]:
for m in kept:
    m.pretty_print()

================================ Human Message =================================

For four people.
================================== Ai Message ==================================

Perfect—**4 guests** noted! 👍

I still need a couple more things to lock in the booking:

1. **Restaurant** – Any preference (cuisine, area, or a specific name you have in mind)?
2. **Time** – What time on Friday works best?
3. **Special requests** – Anything the restaurant should know (e.g., high chair, dietary restrictions, celebration, accessibility needs)?

If you're not sure on the restaurant, tell me what you're in the mood for (Italian, steak, sushi, casual, fancy, etc.) and I can suggest some great options nearby.
================================ Human Message =================================

Around eight in the evening.
================================== Ai Message ==================================

Great—**Friday at 8:00 PM for 4 people** is set! 🕰️

Just one more piece to complete the booking:


The last two exchanges survived, the opening turn is gone. `start_on="human"` stops the list beginning on a reply with no question above it.

### Trim by token budget instead of message count

Message count isn't what you're billed for. `"approximate"` counts tokens without calling anything.

In [ ]:
budgeted = trim_messages(
    full,
    max_tokens=600,
    token_counter="approximate",
    strategy="last",
    start_on="human",
)

budgeted

Compare against `full` above to see if anything was cut. Now set the budget far too low, on purpose.

In [ ]:
trim_messages(
    full,
    max_tokens=200,
    token_counter="approximate",
    strategy="last",
    start_on="human",
)

[HumanMessage(content='Around eight in the evening.', additional_kwargs={}, response_metadata={}, id='c9d00eb5-f440-4bb0-996a-6ab39d192d7f'),
 AIMessage(content='Great—**Friday at 8:00 PM for 4 people** is set! 🕰️\n\nJust one more piece to complete the booking:\n\n- **Which restaurant** (name, area, or city)?\n\nIf you let me know the restaurant (or any cuisine/area preferences), I can check availability and finalize the reservation for you. Also, do you have any special requests I should mention to the restaurant (e.g., high chair, dietary restrictions, celebration, accessibility needs)?', additional_kwargs={'reasoning_content': "The user wants to book a table for Friday at around 8 PM for 4 people. I still need the restaurant name/location. I should ask for that. Also need to know if there are any special requests. I'll respond by asking for the restaurant details and confirming the special requests. Let's craft a helpful response.", 'reasoning_details': [{'type': 'reasoning.text', '

An empty list, no error, no warning. If one exchange doesn't fit the budget, nothing fits, and you've sent the model a conversation with no conversation in it. A too-small budget looks like this: silent.

Trimming's cost is **the beginning of the conversation**, the name from turn one is gone. Right when only recent turns matter, wrong the moment something early matters late.

`trim_messages` also takes `include_system=True`, which pins a system message so it's never cut. It does nothing here, `create_agent` applies `system_prompt` at call time rather than storing it in the list. It matters when you keep your own list, as in section 2.

## 8. Fix two: compress old turns with SummarizationMiddleware

Replace old messages with a short summary instead of dropping them. Middleware does this at a threshold you pick.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

summarising = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a helpful restaurant booking assistant.",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 6),
            keep=("messages", 2),
        )
    ],
    checkpointer=InMemorySaver(),
)

`trigger` is when it fires, `keep` is how much recent conversation survives untouched. Everything older compresses into one summary message.

In [ ]:
sum_cfg = {"configurable": {"thread_id": "summary-demo"}}

def chat_sum(text):
    # time.sleep(6)
    result = summarising.invoke({"messages": [HumanMessage(text)]}, config=sum_cfg)
    return result["messages"][-1].text

chat_sum("My name is Priya and I use a wheelchair.")
chat_sum("I want a table on Friday.")
chat_sum("For four people.")
chat_sum("Around eight in the evening.")
chat_sum("Somewhere quiet, please.")

In [ ]:
summarising.get_state(sum_cfg).values["messages"]

[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nUser Priya (wheelchair user) aims to book an accessible restaurant table for Friday for a group of four, with the AI assisting by prioritizing step-free entrances, accessible restrooms, and wheelchair-friendly seating.\n\n## SUMMARY\n- User disclosed wheelchair use and explicit need for accessibility features.\n- Goal: secure a table for 4 people on Friday.\n- AI committed to filtering for step-free entrances, accessible restrooms, and suitable seating.\n- Missing critical details: city/neighborhood, preferred cuisine or vibe, and specific time on Friday.\n- No decisions or rejections beyond the focus on accessibility; conversation currently pending those missing inputs.\n\n## ARTIFACTS\nNo files, documents, or resources were created, modified, or accessed during this conversation. All interaction was limited to textual messages within the chat interface.\n\n## NEXT STEPS\n- Request the missing 

Look at how many messages are being kept, against how many turns you had.

In [ ]:
# for m in summarising.get_state(sum_cfg).values["messages"]:
#     m.pretty_print()

Summarising costs **detail and an extra API call**. Exact words are gone, replaced by a paraphrase that might drop what you needed, but early facts survive in some form, which trimming couldn't do.

## 9. Long-term memory: facts that outlive the conversation

Everything so far is short-term: one conversation, one `thread_id`. Long-term memory is for facts that outlive the conversation, Priya using a wheelchair is a fact about Priya, still true in March.

The tool is a **store**: facts keyed by who they belong to.

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

store.put(("profile", "priya"), "access", {"note": "Priya uses a wheelchair"})
store.put(("profile", "priya"), "diet", {"note": "Priya is vegetarian"})

[item.value for item in store.search(("profile", "priya"))]

[{'note': 'Priya uses a wheelchair'}, {'note': 'Priya is vegetarian'}]

In [ ]:
store

`("profile", "priya")` is a namespace, a labelled drawer. `"access"` is the key inside it. Two users, two drawers, nothing crosses over.

## 10. Wiring long-term memory into the assistant

The model only reads messages, so facts go into the system prompt, rebuilt on every call.

In [ ]:
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def with_profile(request):
    facts = request.runtime.store.search(("profile", "priya"))
    notes = [item.value["note"] for item in facts]
    return "You are a restaurant booking assistant. Known about this guest:\n" + "\n".join(notes)

In [ ]:
remembering = create_agent(
    model=model,
    tools=[],
    middleware=[with_profile],
    checkpointer=InMemorySaver(),
    store=store,
)

A **brand new** `thread_id`: short-term memory is completely empty. No history, nothing.

In [ ]:
time.sleep(6)

fresh = {"configurable": {"thread_id": "a-conversation-that-never-happened"}}

result = remembering.invoke(
    {"messages": [HumanMessage("Book me a table for Friday.")]},
    config=fresh,
)

result

Two messages in the whole conversation, and it still knew.

- **Short-term memory:** what was said in *this* conversation.
- **Long-term memory:** what's true about *this person*, in every conversation.

Mixing them up is how an assistant forgets your name every morning, or brings up last year's complaint at the wrong moment.

## 11. Save memory to disk with SqliteSaver

`InMemorySaver` is a Python dictionary, restart the kernel and every conversation is gone. Swap it for a file-backed checkpointer and nothing else about your code changes.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

connection = sqlite3.connect("day06_memory.db", check_same_thread=False)
saver = SqliteSaver(connection)
saver.setup()

durable = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a helpful restaurant booking assistant.",
    checkpointer=saver,
)

In [ ]:
# time.sleep(6)

response = durable.invoke(
    {"messages": [HumanMessage("My name is Priya. Remember it.")]},
    config={"configurable": {"thread_id": "on-disk"}},
)

response

In [ ]:
import os

os.path.getsize("day06_memory.db")

4096

That file is on disk now. Restart the kernel, rebuild the agent against the same file, ask for the name on thread `on-disk`, still there.

Same agent code. Only the checkpointer changed.